In [1]:
import os
import json
import torch
from PIL import Image
from tqdm import tqdm
from transformers import Blip2Processor, Blip2ForConditionalGeneration

# ─────────────────────────────────────────
# 1. GPU Check
# ─────────────────────────────────────────
print(f"GPUs available: {torch.cuda.device_count()}")
for i in range(torch.cuda.device_count()):
    print(f"  GPU {i}: {torch.cuda.get_device_name(i)} — "
          f"{torch.cuda.get_device_properties(i).total_memory / 1e9:.1f} GB")

# ─────────────────────────────────────────
# 2. Paths
# ─────────────────────────────────────────
cropped_dir    = '/kaggle/input/datasets/hades199/3c-yolo-cropped-images'
caption_output = '/kaggle/working/captions.json'

# ─────────────────────────────────────────
# 3. Load BLIP-2 (Frozen — as per project spec)
# ─────────────────────────────────────────
print("\nLoading BLIP-2...")

processor = Blip2Processor.from_pretrained(
    "Salesforce/blip2-opt-2.7b",
    use_fast=False      # silences BlipImageProcessor warning
)

model = Blip2ForConditionalGeneration.from_pretrained(
    "Salesforce/blip2-opt-2.7b",
    torch_dtype=torch.float16,
    device_map="auto"   # fixes language_model device_map warning
)
model.eval()
print("BLIP-2 ready!\n")

# ─────────────────────────────────────────
# 4. Prompt
# ─────────────────────────────────────────
CAPTION_PROMPT = (
    "Question: You are a fashion product cataloger. "
    "Describe this clothing item in one sentence covering: "
    "garment type, primary color, pattern or texture, fit or silhouette, "
    "and one or two notable design details such as collar style, "
    "sleeve length, or embellishments. "
    "Answer:"
)

# ─────────────────────────────────────────
# 5. Resume Support
# ─────────────────────────────────────────
if os.path.exists(caption_output):
    with open(caption_output, 'r') as f:
        captions = json.load(f)
    print(f"Resuming — {len(captions)} captions already done.")
else:
    captions = {}

# ─────────────────────────────────────────
# 6. Collect All Images
# ─────────────────────────────────────────
all_images = []
for root, dirs, files in os.walk(cropped_dir):
    for fname in files:
        if fname.lower().endswith(('.jpg', '.jpeg', '.png')):
            full_path = os.path.join(root, fname)
            rel_key   = os.path.relpath(full_path, cropped_dir)
            all_images.append((rel_key, full_path))

print(f"Total images : {len(all_images)}")
print(f"Remaining    : {len(all_images) - len(captions)}\n")

# ─────────────────────────────────────────
# 7. Batch Processing
# ─────────────────────────────────────────
BATCH_SIZE = 32
SAVE_EVERY = 500

# Find which device the model's input side is on
input_device = model.device if hasattr(model, 'device') else "cuda:0"

def process_batch(batch_keys, batch_images, captions):
    inputs = processor(
        images=batch_images,
        text=[CAPTION_PROMPT] * len(batch_images),
        return_tensors="pt",
        padding=True
    ).to(input_device, torch.float16)

    with torch.no_grad():
        generated_ids = model.generate(
            **inputs,
            max_new_tokens=75,
            num_beams=4,
            repetition_penalty=1.3,
            length_penalty=1.0,
            early_stopping=True
        )

    decoded = processor.batch_decode(generated_ids, skip_special_tokens=True)

    for key, text in zip(batch_keys, decoded):
        caption = text.replace(CAPTION_PROMPT, "").strip()
        caption = caption.lstrip(":.").strip()
        captions[key] = caption

# ─────────────────────────────────────────
# 8. Main Loop
# ─────────────────────────────────────────
batch_keys   = []
batch_images = []

for i, (rel_key, full_path) in enumerate(tqdm(all_images, desc="Captioning")):

    if rel_key in captions:
        continue

    try:
        img = Image.open(full_path).convert("RGB")
    except Exception:
        captions[rel_key] = "a clothing item"
        continue

    batch_keys.append(rel_key)
    batch_images.append(img)

    if len(batch_keys) == BATCH_SIZE:
        process_batch(batch_keys, batch_images, captions)
        batch_keys, batch_images = [], []

    if (i + 1) % SAVE_EVERY == 0:
        with open(caption_output, 'w') as f:
            json.dump(captions, f, indent=2)
        print(f"\n💾 Checkpoint: {len(captions)} captions saved.")

# Last partial batch
if batch_keys:
    process_batch(batch_keys, batch_images, captions)

# ─────────────────────────────────────────
# 9. Final Save + Sanity Check
# ─────────────────────────────────────────
with open(caption_output, 'w') as f:
    json.dump(captions, f, indent=2)

print(f"\n✅ Done! {len(captions)} captions saved to {caption_output}")

print("\n── Sample Captions ──────────────────────────────")
for key, cap in list(captions.items())[:5]:
    print(f"  {os.path.basename(key)}")
    print(f"  → {cap}\n")

GPUs available: 2
  GPU 0: Tesla T4 — 15.6 GB
  GPU 1: Tesla T4 — 15.6 GB

Loading BLIP-2...


processor_config.json:   0%|          | 0.00/68.0 [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/432 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json:   0%|          | 0.00/882 [00:00<?, ?B/s]

...


💾 Checkpoint: 52000 captions saved.


The `language_model` is not in the `hf_device_map` dictionary and you are running your script in a multi-GPU environment. this may lead to unexpected behavior when using `accelerate`. Please pass a `device_map` that contains `language_model` to remove this warning. Please refer to https://github.com/huggingface/blog/blob/main/accelerate-large-models.md for more details on creating a `device_map` for large models.

Captioning:  99%|█████████▉| 52057/52712 [6:08:41<03:20,  3.27it/s]The `language_model` is not in the `hf_device_map` dictionary and you are running your script in a multi-GPU environment. this may lead to unexpected behavior when using `accelerate`. Please pass a `device_map` that contains `language_model` to remove this warning. Please refer to https://github.com/huggingface/blog/blob/main/accelerate-large-models.md for more details on creating a `device_map` for large models.

Captioning:  99%|█████████▉| 52080/52712 [6:08:57<04:24,  2.39it/s]The `language_model` is not in


💾 Checkpoint: 52480 captions saved.



Captioning: 100%|█████████▉| 52542/52712 [6:12:12<00:51,  3.33it/s]The `language_model` is not in the `hf_device_map` dictionary and you are running your script in a multi-GPU environment. this may lead to unexpected behavior when using `accelerate`. Please pass a `device_map` that contains `language_model` to remove this warning. Please refer to https://github.com/huggingface/blog/blob/main/accelerate-large-models.md for more details on creating a `device_map` for large models.

Captioning: 100%|█████████▉| 52574/52712 [6:12:25<00:36,  3.77it/s]The `language_model` is not in the `hf_device_map` dictionary and you are running your script in a multi-GPU environment. this may lead to unexpected behavior when using `accelerate`. Please pass a `device_map` that contains `language_model` to remove this warning. Please refer to https://github.com/huggingface/blog/blob/main/accelerate-large-models.md for more details on creating a `device_map` for large models.

Captioning: 100%|█████████▉| 


✅ Done! 52712 captions saved to /kaggle/working/captions.json

── Sample Captions ──────────────────────────────
  01_2_side.jpg
  → the olive green jacket

  01_3_back.jpg
  → The jacket is an olive green corduroy jacket with a button-down collar. It is worn over a pair of blue jeans and a pair of white sneakers.

  01_6_flat.jpg
  → This is a light blue denim jean pant with a distressed look.

  01_1_front.jpg
  → an olive green parka jacket

  01_4_full.jpg
  → A dark green jacket with a blue shirt and jeans

